In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
model_name = "facebook/esm2_t6_8M_UR50D"

tokenizer = AutoTokenizer.from_pretrained(model_name)
esm_model = AutoModel.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
esm_model = esm_model.to(device)
esm_model.eval()

In [ ]:
y_train_esm = y_train
y_test_esm  = y_test

X_train_esm = get_embeddings_batch(X_train_text, tokenizer, esm_model, device)
X_test_esm  = get_embeddings_batch(X_test_text,  tokenizer, esm_model, device)

y_train_esm = train_sample["label"].values
y_test_esm = test_sample["label"].values

def get_embedding(seq):
    inputs = tokenizer(seq, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = esm_model(**inputs)

    return outputs.last_hidden_state.mean(dim=1).cpu().numpy().flatten()

X_train_esm = np.array([get_embedding(seq) for seq in train_sample["seq"]])
X_test_esm = np.array([get_embedding(seq) for seq in test_sample["seq"]])

In [ ]:
esm_model_ridge = Ridge()
esm_model_ridge.fit(X_train_esm, y_train_esm)

esm_preds = esm_model_ridge.predict(X_test_esm)

esm_mae = mean_absolute_error(y_test_esm, esm_preds)
esm_r2 = r2_score(y_test_esm, esm_preds)

print("ESM MAE:", esm_mae)
print("ESM R2:", esm_r2)